# Chapter 1: Python Fundamentals for AI
## Notebook 03 - Advanced

The final notebook in Chapter 1. Here we cover the patterns and techniques that separate beginner Python from professional AI code.

**What you'll learn:**
- Object-Oriented Programming for AI
- File I/O (CSV, JSON, text)
- Generators and iterators
- Decorators
- Capstone project: build a mini ML experiment tracker

**Time estimate:** 2 hours

---
*Generated by Berta AI | Created by Luigi Pascal Rondanini*

## 1. Object-Oriented Programming for AI

Classes are how PyTorch, TensorFlow, scikit-learn, and every major AI framework organize their code. Understanding OOP is essential.

In [ ]:
import random
import math
from datetime import datetime


class NeuralLayer:
    """A simplified neural network layer to demonstrate OOP concepts."""
    
    def __init__(self, input_size, output_size, activation="relu"):
        self.input_size = input_size
        self.output_size = output_size
        self.activation = activation
        
        # Xavier initialization (simplified)
        scale = math.sqrt(2.0 / (input_size + output_size))
        self.weights = [
            [random.gauss(0, scale) for _ in range(output_size)]
            for _ in range(input_size)
        ]
        self.bias = [0.0] * output_size
    
    def forward(self, inputs):
        """Compute the forward pass."""
        output = list(self.bias)
        for i, x in enumerate(inputs):
            for j in range(self.output_size):
                output[j] += x * self.weights[i][j]
        
        if self.activation == "relu":
            output = [max(0, v) for v in output]
        elif self.activation == "sigmoid":
            output = [1 / (1 + math.exp(-min(max(v, -500), 500))) for v in output]
        
        return output
    
    @property
    def num_parameters(self):
        return self.input_size * self.output_size + self.output_size
    
    def __repr__(self):
        return (f"NeuralLayer({self.input_size} -> {self.output_size}, "
                f"activation={self.activation}, params={self.num_parameters})")


# Build a small network
random.seed(42)
layer1 = NeuralLayer(4, 8, "relu")
layer2 = NeuralLayer(8, 3, "sigmoid")

print(f"Layer 1: {layer1}")
print(f"Layer 2: {layer2}")
print(f"Total params: {layer1.num_parameters + layer2.num_parameters}")

# Forward pass
sample_input = [0.5, -0.3, 0.8, 0.1]
hidden = layer1.forward(sample_input)
output = layer2.forward(hidden)

print(f"\nInput:  {sample_input}")
print(f"Hidden: {[f'{h:.4f}' for h in hidden]}")
print(f"Output: {[f'{o:.4f}' for o in output]}")
print(f"Predicted class: {output.index(max(output))}")

In [ ]:
class SimpleNetwork:
    """A simple sequential neural network."""
    
    def __init__(self, *layers):
        self.layers = list(layers)
    
    def forward(self, inputs):
        x = inputs
        for layer in self.layers:
            x = layer.forward(x)
        return x
    
    @property
    def total_parameters(self):
        return sum(layer.num_parameters for layer in self.layers)
    
    def summary(self):
        print(f"{'Layer':>10} {'Shape':>15} {'Activation':>12} {'Params':>8}")
        print("-" * 50)
        for i, layer in enumerate(self.layers):
            shape = f"{layer.input_size} -> {layer.output_size}"
            print(f"{'Layer ' + str(i+1):>10} {shape:>15} {layer.activation:>12} {layer.num_parameters:>8}")
        print("-" * 50)
        print(f"{'Total':>10} {'':>15} {'':>12} {self.total_parameters:>8}")


random.seed(42)
net = SimpleNetwork(
    NeuralLayer(10, 64, "relu"),
    NeuralLayer(64, 32, "relu"),
    NeuralLayer(32, 5, "sigmoid"),
)

net.summary()

sample = [random.uniform(-1, 1) for _ in range(10)]
prediction = net.forward(sample)
print(f"\nPrediction: {[f'{p:.4f}' for p in prediction]}")

## 2. File I/O

Reading and writing data files is a daily task in AI. CSV for tabular data, JSON for configs and APIs, text files for NLP datasets.

In [ ]:
import csv
import json
import os

# Generate and write a CSV dataset
random.seed(42)
data_dir = os.path.join(os.path.dirname(os.getcwd()), "datasets") if os.path.exists("../datasets") else "/tmp"
os.makedirs(data_dir, exist_ok=True)
csv_path = os.path.join(data_dir, "sample_data.csv")

headers = ["id", "feature_1", "feature_2", "feature_3", "label"]
rows = []
for i in range(50):
    f1 = round(random.gauss(0, 1), 4)
    f2 = round(random.gauss(0.5, 0.8), 4)
    f3 = round(random.uniform(0, 10), 4)
    label = "positive" if (f1 + f2 * 0.5 + f3 * 0.1) > 0.5 else "negative"
    rows.append([i, f1, f2, f3, label])

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows to {csv_path}")

# Read it back
with open(csv_path, 'r') as f:
    reader = csv.DictReader(f)
    loaded_data = list(reader)

print(f"\nLoaded {len(loaded_data)} rows")
print(f"Columns: {list(loaded_data[0].keys())}")
print(f"First row: {loaded_data[0]}")

In [ ]:
# JSON: configs, API responses, experiment logs
experiment = {
    "name": "transformer-v1",
    "timestamp": datetime.now().isoformat(),
    "config": {
        "model": "bert-base",
        "learning_rate": 3e-5,
        "epochs": 5,
        "batch_size": 16,
    },
    "results": {
        "accuracy": 0.923,
        "f1_score": 0.917,
        "training_time_seconds": 3420,
    },
    "tags": ["nlp", "classification", "bert"],
}

json_path = os.path.join(data_dir, "experiment.json")
with open(json_path, 'w') as f:
    json.dump(experiment, f, indent=2)

print(f"Experiment saved to {json_path}")
print(json.dumps(experiment, indent=2))

## 3. Generators and Iterators

Generators produce values on-the-fly without loading everything into memory. Essential for processing large datasets that don't fit in RAM.

In [ ]:
def data_batch_generator(data, batch_size=8, shuffle=True):
    """Generate batches from a dataset (like PyTorch DataLoader)."""
    indices = list(range(len(data)))
    if shuffle:
        random.shuffle(indices)
    
    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start:start + batch_size]
        batch = [data[i] for i in batch_indices]
        yield batch


# Simulate a dataset of 25 items
random.seed(42)
dataset = [f"sample_{i}" for i in range(25)]

print("Iterating through batches:")
for batch_idx, batch in enumerate(data_batch_generator(dataset, batch_size=8)):
    print(f"  Batch {batch_idx}: {len(batch)} items -> {batch[:3]}...")

print(f"\nTotal items: {len(dataset)}")
print(f"Batch size: 8")
print(f"Number of batches: {math.ceil(len(dataset) / 8)}")

## 4. Decorators

Decorators wrap functions to add behavior — like timing, logging, caching, and validation. You'll see them in every major framework.

In [ ]:
import time
import functools


def timer(func):
    """Measure execution time of a function."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"  [{func.__name__}] completed in {elapsed:.4f}s")
        return result
    return wrapper


def retry(max_attempts=3, delay=0.1):
    """Retry a function if it raises an exception."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts:
                        raise
                    print(f"  Attempt {attempt} failed: {e}. Retrying...")
                    time.sleep(delay)
        return wrapper
    return decorator


@timer
def process_data(n):
    """Simulate a data processing task."""
    return sum(i ** 2 for i in range(n))


@retry(max_attempts=3)
def unreliable_api_call():
    """Simulate a flaky API."""
    if random.random() < 0.6:
        raise ConnectionError("API timeout")
    return {"status": "success", "data": [1, 2, 3]}


print("Timer decorator:")
result = process_data(1_000_000)
print(f"  Result: {result:,}")

print("\nRetry decorator:")
random.seed(42)
try:
    response = unreliable_api_call()
    print(f"  Response: {response}")
except ConnectionError:
    print("  All attempts failed.")

## 5. Capstone: Mini Experiment Tracker

Let's build a small but functional experiment tracker — combining everything from all three notebooks.

In [ ]:
class ExperimentTracker:
    """Track ML experiments with logging, metrics, and comparison."""
    
    def __init__(self, project_name):
        self.project_name = project_name
        self.experiments = []
        self.created_at = datetime.now()
    
    def log_experiment(self, name, config, metrics, tags=None):
        """Log a new experiment."""
        experiment = {
            "id": len(self.experiments) + 1,
            "name": name,
            "config": config,
            "metrics": metrics,
            "tags": tags or [],
            "timestamp": datetime.now().isoformat(),
        }
        self.experiments.append(experiment)
        return experiment["id"]
    
    def get_best(self, metric="accuracy", higher_is_better=True):
        """Find the best experiment by a given metric."""
        if not self.experiments:
            return None
        key = lambda e: e["metrics"].get(metric, 0)
        return max(self.experiments, key=key) if higher_is_better else min(self.experiments, key=key)
    
    def compare(self, metric="accuracy"):
        """Print a comparison table of all experiments."""
        sorted_exps = sorted(
            self.experiments,
            key=lambda e: e["metrics"].get(metric, 0),
            reverse=True
        )
        
        print(f"\n{'#':>3} {'Name':>20} {'Accuracy':>10} {'Loss':>10} {'F1':>10} {'Time(s)':>10}")
        print("-" * 67)
        for exp in sorted_exps:
            m = exp["metrics"]
            print(f"{exp['id']:>3} {exp['name']:>20} "
                  f"{m.get('accuracy', 0):>10.4f} "
                  f"{m.get('loss', 0):>10.4f} "
                  f"{m.get('f1_score', 0):>10.4f} "
                  f"{m.get('training_time', 0):>10.1f}")
    
    def summary(self):
        """Print project summary."""
        print(f"\nProject: {self.project_name}")
        print(f"Experiments: {len(self.experiments)}")
        if self.experiments:
            best = self.get_best()
            print(f"Best (accuracy): {best['name']} ({best['metrics']['accuracy']:.2%})")
        print()


# Use the tracker
tracker = ExperimentTracker("text-classification")

experiments = [
    ("logistic-regression", {"model": "sklearn-lr", "C": 1.0}, 
     {"accuracy": 0.82, "loss": 0.41, "f1_score": 0.79, "training_time": 2.1}),
    ("random-forest", {"model": "sklearn-rf", "n_estimators": 100},
     {"accuracy": 0.87, "loss": 0.32, "f1_score": 0.85, "training_time": 15.3}),
    ("bert-base-ft", {"model": "bert-base", "lr": 3e-5, "epochs": 3},
     {"accuracy": 0.93, "loss": 0.19, "f1_score": 0.91, "training_time": 1240.0}),
    ("distilbert-ft", {"model": "distilbert", "lr": 5e-5, "epochs": 5},
     {"accuracy": 0.91, "loss": 0.22, "f1_score": 0.89, "training_time": 680.0}),
    ("gpt4-zeroshot", {"model": "gpt-4", "prompt": "classify"},
     {"accuracy": 0.88, "loss": 0.28, "f1_score": 0.86, "training_time": 0.0}),
]

for name, config, metrics in experiments:
    tracker.log_experiment(name, config, metrics)

tracker.summary()
tracker.compare()

print("\n--- Best by different metrics ---")
best_acc = tracker.get_best("accuracy")
best_speed = tracker.get_best("training_time", higher_is_better=False)
print(f"Best accuracy:   {best_acc['name']} ({best_acc['metrics']['accuracy']:.2%})")
print(f"Fastest:         {best_speed['name']} ({best_speed['metrics']['training_time']:.1f}s)")

## Chapter 1 Complete!

Congratulations! You've covered all the Python fundamentals needed for AI work:

- **Notebook 01**: Variables, types, strings, control flow, loops, comprehensions
- **Notebook 02**: Collections, functions, error handling, modules, data pipelines
- **Notebook 03**: OOP, file I/O, generators, decorators, experiment tracking

### Ready for What's Next?

Now try the **exercises** in `exercises/exercises.py` to solidify your skills.

Then move on to **Chapter 2: Data Structures & Algorithms** where you'll learn about algorithmic thinking and complexity analysis.

---

*Generated by Berta AI | Created by Luigi Pascal Rondanini*